# Enhanced Index Tracking with MIQP

This notebook implements a Mixed-Integer Quadratic Programming (MIQP)
model for enhanced index tracking of the Swiss Market Index (SMI).

The model minimises tracking error variance while requiring a minimum
expected excess return relative to the SMI.

In [ ]:
#Import
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB, quicksum

In [ ]:
#Import Data, drop the columns Date and SMI Index
df=pd.read_csv("data/rebalancing/prices_smi_20240412.csv", sep=";", header=0)
df=df.drop(labels=["Date","SMI Index"],axis=1)

# Define our T
T=df.shape[0]-1

In [ ]:
#Calculate the returns
returns= df.select_dtypes(include=['number']).pct_change()

# remove first row with NaN values
returns=returns.iloc[1:]

In [ ]:
#Calculate the covariance matrix
sigma=returns.cov()

In [ ]:
#Create an empty model

model = gp.Model('portfolio')
model.setParam("FeasibilityTol", 1e-06)

In [ ]:
# U: complete SMI stock universe
U = df.columns.values

# I: stocks considered for portfolio construction
# All SMI constituents are included in this implementation.
I = U

In [ ]:
#Read in the weights
weights = pd.read_csv("data/rebalancing/close_smi_20240412.csv", sep=";", header=0, index_col=[0])

# Keep only the columns with names "Instrument" or "Weight"
selected_columns = weights.loc[:, weights.columns.isin(["Instrument_Name", "Weight"])]

#Reformate all the stock names to the "normal" form instead of all upper-case
# Custom function to format the names
def format_instrument_name(name):
    words = name.split()
    formatted_words = []
    for word in words:
        if word.upper() in ['ABB', 'UBS', 'AG', 'SA']:
            formatted_words.append(word.upper())
        else:
            formatted_words.append(word.capitalize())
    return ' '.join(formatted_words)

# Apply the custom function to the "Instrument_Name" column
selected_columns.loc[:, 'Instrument_Name'] = selected_columns['Instrument_Name'].apply(format_instrument_name)

#Set Instrument_Name as Index
selected_columns.set_index("Instrument_Name",inplace=True)
weights=selected_columns
#Transpose the Matrix
weights=weights.transpose()
# Convert weights to a dictionary 
weight_records = weights.to_dict("records")
w = weight_records[0]
# parse weights in the dictionary to floats and divide by 100 to get percent values
for stock, weight in w.items():
    w[stock] = float(weight) / 100
w

In [ ]:
# Load current portfolio holdings
portfolio = pd.read_csv(
    "data/rebalancing/current_portfolio_holdings.csv",
    sep=";",
    header=0,
    index_col=[0]
).transpose()

# Calculate current market value of each holding
portfolio_value = portfolio.iloc[0] * df.iloc[-1]

# Total portfolio value used as optimisation budget
C = portfolio_value.sum()

print(f"Portfolio value: CHF {C:,.2f}")

In [ ]:
## Model parameters

alpha = 0.02       # Minimum expected excess return
epsilon = 0.01     # Minimum portfolio weight
delta = 0.30       # Maximum portfolio weight
k = 10             # Number of selected stocks

In [ ]:
# Continuous variables: number of shares held
x = model.addVars(
    U,
    name="x",
    lb=0.0
)

# Binary variables: stock selection
z = model.addVars(
    U,
    name="z",
    vtype=GRB.BINARY
)

In [ ]:
# Annualised historical mean returns
r_bar = returns.mean() * 240

## MIQP Formulation

The objective minimises the Tracking Error Variance (TEV) between the
portfolio and the SMI.

The model additionally imposes:

- a full-investment budget constraint,
- a minimum expected excess return,
- minimum and maximum portfolio weights,
- and a cardinality constraint.

In [ ]:
#Objective Function
TEV = quicksum(
    sigma[i][j] * (((df[i][T] * x[i]) / C) - w[i]) * (((df[j][T] * x[j]) / C) - w[j])
    for i in I
    for j in I
) - 2 * quicksum(
    sigma[i][j] * (((df[i][T] * x[i]) / C) * w[i] - w[i] * w[j])
    for i in I
    for j in set(U) - set(I)
) + quicksum(
    sigma[j][j] * w[i] * w[j]
    for i in set(U) - set(I)
    for j in set(U) - set(I)
)

In [ ]:
# Objective: minimise tracking error variance
model.setObjective(TEV * 1e6, GRB.MINIMIZE)


# Full investment
model.addConstr(quicksum(df[i][T]*x[i] for i in I)==C, 'budget constraint')

# Minimum expected excess return
model.addConstr(quicksum(((df[i][T]*x[i])/C)*r_bar[i] for i in I) - quicksum(w[i]*r_bar[i] for i in U)>=alpha, 'excess return')

# Minimum and maximum portfolio weights
model.addConstrs((epsilon*z[i] <=df[i][T]*x[i]/C for i in I), "min investment")
model.addConstrs((delta*z[i] >=df[i][T]*x[i]/C for i in I), "max investment")


# Exactly 10 stocks are selected
model.addConstr(quicksum(z[i] for i in I)==k, 'cardinality constraint')

# Verify model formulation
model.write('portfolio_selection_optimization.lp')

# Solve
model.optimize()

In [ ]:
# Extract optimised portfolio
result = pd.DataFrame(index=I)

for i in I:
    result.loc[i, "Weight"] = (
        x[i].X * df[i][T] / C * 100
    )
    result.loc[i, "Shares"] = x[i].X

result

In [ ]:
# Check portfolio constraints
print(f"Total portfolio weight: {result['Weight'].sum():.2f}%")
print(f"Selected stocks: {(result['Weight'] > 0).sum()}")